# 模块 G · NMF 表达程序（把表达拆成若干个"基因程序"，看每个程序的空间分布）   [Python]

- 对应课程：第18课
- 适用：全转录组平台（Visium、HD、Stereo-seq）；靶向 panel 也能跑，但程序数要少。
- 注意：NMF 要求非负输入，用 log 标准化后的值（不能用 scale 之后的值）；HD 数据点多时先下采样拟合，再投影全部数据。

**对应的流程脚本**：`scripts/G_special/nmf_programs.py`（代码相同，流程运行的是 .py）

**用法**：先把下面第一个代码格里的 `project_dir` 改成项目目录，然后从上到下逐格运行。

In [ ]:
# 步骤：切换到项目目录（里面有 config.yaml、samples.tsv 和 results/）
# 输入：project_dir（改成自己的项目目录）
# 输出：无（之后的代码都以项目目录为当前目录）
import os
project_dir = "."      # 例如 "/data/my_visium_project"
os.chdir(project_dir)
os.getcwd()

## 1. 导入包

In [ ]:
# 步骤：导入包
# 输入：无
# 输出：无
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from sklearn.decomposition import NMF
import squidpy as sq
import yaml

## 2. 读取配置，设定程序个数

In [ ]:
# 步骤：读取配置，设定程序个数
# 输入：config.yaml
# 输出：cfg、out_dir、n_programs、n_fit
cfg = yaml.safe_load(open("config.yaml"))
out_dir = Path(cfg["out_dir"])
n_programs = 10      # 可以试 6 / 10 / 15，看程序是否可解释
n_fit = 50000        # 拟合时最多用多少个 spot / 细胞

这一步得到：cfg、out_dir、n_programs、n_fit

## 3. 读入数据，只用高变基因

In [ ]:
# 步骤：读入数据，只用高变基因
# 输入：results/05_niche.h5ad
# 输出：expr（spot x 高变基因，log 标准化值）
adata = sc.read_h5ad(out_dir / "05_niche.h5ad")
expr = adata[:, adata.var["highly_variable"]].copy()
expr.shape

这一步得到：expr（spot x 高变基因，log 标准化值）

## 4. 在下采样的数据上拟合 NMF，再把全部数据投影上去

In [ ]:
# 步骤：在下采样的数据上拟合 NMF，再把全部数据投影上去
# 输入：expr
# 输出：usage（spot x 程序）、loadings（程序 x 基因）
fit_cells = sc.pp.subsample(expr, n_obs=min(n_fit, expr.n_obs), random_state=cfg["seed"], copy=True)
nmf = NMF(n_components=n_programs, init="nndsvda", max_iter=500, random_state=cfg["seed"])
nmf.fit(fit_cells.X)
programs = [f"NMF_{i + 1}" for i in range(n_programs)]
usage = pd.DataFrame(nmf.transform(expr.X), index=adata.obs_names, columns=programs)
loadings = pd.DataFrame(nmf.components_, index=programs, columns=expr.var_names)

这一步得到：usage（spot x 程序）、loadings（程序 x 基因）

## 5. 保存每个程序权重最高的 20 个基因

In [ ]:
# 步骤：保存每个程序权重最高的 20 个基因
# 输入：loadings
# 输出：表 results/nmf_top_genes.csv
top_genes = pd.DataFrame({p: loadings.loc[p].nlargest(20).index for p in programs})
top_genes.to_csv(out_dir / "nmf_top_genes.csv", index=False)
top_genes.head(10)

这一步得到：表 results/nmf_top_genes.csv

## 6. 画每个程序的空间分布（按行归一化后的使用比例）

In [ ]:
# 步骤：画每个程序的空间分布（按行归一化后的使用比例）
# 输入：usage
# 输出：图 results/spatial_nmf.png
adata.obs[programs] = usage.div(usage.sum(axis=1), axis=0).fillna(0)
sq.pl.spatial_scatter(adata, color=programs, library_key="sample", spatial_key="spatial_um", shape=None, cmap="magma")
plt.savefig(out_dir / "spatial_nmf.png", bbox_inches="tight")

这一步得到：图 results/spatial_nmf.png

## 7. 每个 niche 的程序使用量，帮助解释 niche 的功能

In [ ]:
# 步骤：每个 niche 的程序使用量，帮助解释 niche 的功能
# 输入：adata
# 输出：图 results/nmf_by_niche.png
sc.pl.matrixplot(adata, programs, groupby="niche", standard_scale="var", cmap="Reds", show=False)
plt.savefig(out_dir / "nmf_by_niche.png", bbox_inches="tight")

这一步得到：图 results/nmf_by_niche.png